# iPermit — Face Verification Evaluation (v2)

Measures how well iPermit's face-matching pipeline separates **the same person** (genuine pairs)
from **different people** (impostor pairs), on two public celebrity datasets:

| Dataset | Population | Source |
|---|---|---|
| **LFW** (Labeled Faces in the Wild, funneled) | Mostly Western public figures | scikit-learn download |
| **100 Bollywood Celebrity Faces** | South Asian actors/actresses | Kaggle `havingfun/100-bollywood-celebrity-faces` |

### What changed from the first evaluation run
1. **Same pipeline as the backend.** Images are decoded, CLAHE-enhanced (the exact function from
   `backend/app/core/face_preprocessing.py`), then passed to insightface `buffalo_l` at 640×640 —
   the same steps as `backend/app/core/face_engine.py::detect_faces`. The first run skipped CLAHE.
2. **Same protocol for both datasets.** Up to **15 usable images per person**, chosen with a fixed
   random seed. The first run used *every* LFW image, so George W. Bush alone produced ~59% of all
   LFW genuine pairs and the LFW-vs-Bollywood comparison was not like-for-like.
3. **Images with 0 or 2+ faces are skipped** — the backend rejects those too
   (`face_service.py`, `police_service.py`). Skips are counted and reported.
4. **Exact duplicate files are skipped** (by MD5), so a photo can't be matched against itself.
5. **Everything is saved**: embeddings, raw scores, metrics JSON, paper-ready tables and figures.
6. **Uncertainty**: 95% confidence intervals from an *identity-level bootstrap* (pairs from the same
   person are not independent, so resampling people — not pairs — is the correct unit).

### How to run
1. **Runtime → Change runtime type → T4 GPU** (CPU also works, just slower: ~15–25 min).
2. Add two secrets in the 🔑 **Secrets** panel (left sidebar), and switch on *Notebook access*
   for both: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token).
   **Never paste the key into a cell.**
3. **Runtime → Run all.** At the end a zip file `ipermit_face_eval_results.zip` downloads —
   send that back.

> "Accuracy" is deliberately **not** reported: with ~100× more impostor pairs than genuine pairs,
> a system that rejects everything would score >99% accuracy. FAR / FRR / EER / TAR@FAR are the
> standard verification metrics (ISO/IEC 19795-1).

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

SEED = 42
MAX_IMAGES_PER_IDENTITY = 15      # same cap for both datasets
LFW_MIN_IMAGES_PER_IDENTITY = 15  # LFW people with >= 15 photos (96 people)
MATCH_THRESHOLD = 0.42            # backend settings.face_match_threshold
FAR_TARGETS = (1e-3, 1e-4)        # report TAR at these false-accept rates
N_BOOTSTRAP = 1000
THRESHOLD_STEP = 0.001            # same resolution as backend find_equal_error_rate (steps=1000)

SAVE_TO_DRIVE = False             # True = keep results in Google Drive (survives runtime resets)
REUSE_CACHED_EMBEDDINGS = True    # skip re-embedding if this run's embeddings file already exists

# Environment overrides exist only so this notebook can be smoke-tested outside Colab.
MAX_IDENTITIES = int(os.environ.get("EVAL_MAX_IDENTITIES", "0")) or None
WORK_DIR = Path("/content") if IN_COLAB else Path(os.environ.get("EVAL_WORK_DIR", "."))
LFW_DIR = Path(os.environ["EVAL_LFW_DIR"]) if "EVAL_LFW_DIR" in os.environ else None
BOLLYWOOD_DIR = Path(os.environ.get("EVAL_BOLLYWOOD_DIR", WORK_DIR / "bollywood_raw"))

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/ipermit_face_eval")
else:
    OUT_DIR = WORK_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUT_DIR.resolve())

## 2. Install packages (Colab only)

In [ ]:
import subprocess

if IN_COLAB:
    # Colab can ship both the CPU and GPU onnxruntime builds; having both installed breaks the import.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"],
        check=False,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu", "insightface", "kaggle"],
        check=True,
    )

## 3. Kaggle credentials (from Colab Secrets — never hard-coded)

In [ ]:
if IN_COLAB and not BOLLYWOOD_DIR.exists():
    from google.colab import userdata

    try:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception as exc:
        raise RuntimeError(
            "Add KAGGLE_USERNAME and KAGGLE_KEY in the Colab Secrets panel (key icon, left sidebar) "
            "and turn on 'Notebook access' for both."
        ) from exc
    print("Kaggle credentials loaded from Colab Secrets.")

## 4. The face pipeline — identical to the backend

`apply_clahe` is copied verbatim from `backend/app/core/face_preprocessing.py`.
`embed_file` mirrors `backend/app/core/face_engine.py::detect_faces`: decode → CLAHE →
insightface `buffalo_l` (RetinaFace detector + ArcFace `w600k_r50`) at 640×640 → L2-normalised
512-d embedding. Only the detection and recognition models are loaded: the other `buffalo_l`
models (landmarks, gender/age) don't affect the embedding — ArcFace aligns using the detector's
5 keypoints — so the embeddings are identical to the backend's, just faster to compute.

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

try:  # lets onnxruntime find the CUDA libraries that ship with Colab's PyTorch
    import torch  # noqa: F401

    ort.preload_dlls()
except Exception:
    pass


def apply_clahe(image: np.ndarray) -> np.ndarray:
    """Contrast-enhances the L channel of a BGR image in LAB space, before
    face detection."""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l_channel)
    enhanced_lab = cv2.merge((l_enhanced, a_channel, b_channel))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)


from insightface.app import FaceAnalysis

_providers = ["CPUExecutionProvider"]
if "CUDAExecutionProvider" in ort.get_available_providers():
    _providers.insert(0, "CUDAExecutionProvider")
face_app = FaceAnalysis(
    name="buffalo_l", providers=_providers, allowed_modules=["detection", "recognition"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
ACTIVE_PROVIDER = face_app.models["recognition"].session.get_providers()[0]
print("Inference running on:", ACTIVE_PROVIDER)
if ACTIVE_PROVIDER == "CPUExecutionProvider":
    print("  (GPU not in use — results are identical, the run just takes longer.)")


def decode_image(path: Path):
    return cv2.imdecode(np.frombuffer(path.read_bytes(), dtype=np.uint8), cv2.IMREAD_COLOR)


def embed_file(path: Path):
    """Returns (status, embedding, bbox). status is 'ok', 'unreadable', 'no_face' or
    'multiple_faces' — the backend rejects the last three the same way."""
    image = decode_image(path)
    if image is None:
        return "unreadable", None, None
    faces = face_app.get(apply_clahe(image))
    if len(faces) == 0:
        return "no_face", None, None
    if len(faces) > 1:
        return "multiple_faces", None, None
    embedding = faces[0].embedding.astype(np.float32)
    embedding /= np.linalg.norm(embedding)
    return "ok", embedding, tuple(float(v) for v in faces[0].bbox)

## 5. Download the datasets

In [ ]:
import zipfile

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}


def list_identity_folders(root: Path) -> dict:
    """identity name -> sorted image paths, for every folder that directly contains images."""
    identities = {}
    for folder in sorted(p for p in root.rglob("*") if p.is_dir()):
        images = sorted(f for f in folder.iterdir() if f.suffix.lower() in IMAGE_SUFFIXES)
        if not images:
            continue
        if folder.name in identities:
            print(f"  note: '{folder.name}' appears in more than one folder — merging")
        identities.setdefault(folder.name, []).extend(images)
    return identities


# LFW (funneled) — scikit-learn downloads and unpacks it on first use.
if LFW_DIR is None:
    from sklearn.datasets import fetch_lfw_people, get_data_home

    fetch_lfw_people(min_faces_per_person=LFW_MIN_IMAGES_PER_IDENTITY, resize=0.2)
    LFW_DIR = Path(get_data_home()) / "lfw_home" / "lfw_funneled"
lfw_identities = {
    name: files
    for name, files in list_identity_folders(LFW_DIR).items()
    if len(files) >= LFW_MIN_IMAGES_PER_IDENTITY
}
print(f"LFW: {len(lfw_identities)} people with >= {LFW_MIN_IMAGES_PER_IDENTITY} photos")

# 100 Bollywood Celebrity Faces — Kaggle (~1.6 GB).
if not BOLLYWOOD_DIR.exists():
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "havingfun/100-bollywood-celebrity-faces",
         "-p", str(WORK_DIR)],
        check=True,
    )
    with zipfile.ZipFile(WORK_DIR / "100-bollywood-celebrity-faces.zip") as archive:
        archive.extractall(BOLLYWOOD_DIR)
bollywood_identities = list_identity_folders(BOLLYWOOD_DIR)
print(f"Bollywood: {len(bollywood_identities)} people")

## 6. Sample up to 15 usable photos per person and compute embeddings

Each person's photos are shuffled with a fixed seed; photos are embedded in that order until
15 usable ones (exactly one face) are collected. People with fewer than 2 usable photos are
dropped (they can't form a genuine pair).

In [ ]:
import hashlib
import json
import time
from collections import Counter


def build_dataset(label: str, identities: dict, root: Path) -> dict:
    cache = OUT_DIR / f"{label.lower()}_embeddings.npz"
    settings_key = json.dumps({"seed": SEED, "cap": MAX_IMAGES_PER_IDENTITY,
                               "lfw_min": LFW_MIN_IMAGES_PER_IDENTITY, "max_ids": MAX_IDENTITIES})
    data = np.load(cache, allow_pickle=False) if REUSE_CACHED_EMBEDDINGS and cache.exists() else None
    if data is not None and str(data["settings_key"]) != settings_key:
        print(f"{label}: cached embeddings were made with different settings — recomputing")
        data = None
    if data is not None:
        print(f"{label}: reusing cached embeddings from {cache.name}")
        return {
            "label": label,
            "root": root,
            "embeddings": data["embeddings"],
            "labels": data["labels"],
            "paths": list(data["paths"]),
            "bboxes": data["bboxes"],
            "identity_names": list(data["identity_names"]),
            "status_counts": json.loads(str(data["status_counts"])),
            "identities_available": int(data["identities_available"]),
        }

    rng = np.random.default_rng(SEED)
    names = sorted(identities)[:MAX_IDENTITIES] if MAX_IDENTITIES else sorted(identities)
    embeddings, labels, paths, bboxes, kept_names = [], [], [], [], []
    status_counts = Counter()
    seen_hashes = set()
    started = time.time()

    for n, name in enumerate(names, 1):
        files = identities[name]
        usable = []
        for index in rng.permutation(len(files)):
            if len(usable) == MAX_IMAGES_PER_IDENTITY:
                break
            path = files[index]
            digest = hashlib.md5(path.read_bytes()).hexdigest()
            if digest in seen_hashes:
                status_counts["duplicate_file"] += 1
                continue
            seen_hashes.add(digest)
            status, embedding, bbox = embed_file(path)
            status_counts[status] += 1
            if status == "ok":
                usable.append((embedding, path, bbox))
        if len(usable) < 2:
            status_counts["people_dropped_under_2_usable"] += 1
            continue
        for embedding, path, bbox in usable:
            embeddings.append(embedding)
            labels.append(len(kept_names))
            paths.append(str(path.relative_to(root)))
            bboxes.append(bbox)
        kept_names.append(name)
        if n % 10 == 0 or n == len(names):
            print(f"  {label}: {n}/{len(names)} people, {len(embeddings)} images, "
                  f"{time.time() - started:.0f}s")

    dataset = {
        "label": label,
        "root": root,
        "embeddings": np.stack(embeddings),
        "labels": np.array(labels, dtype=np.int32),
        "paths": paths,
        "bboxes": np.array(bboxes, dtype=np.float32),
        "identity_names": kept_names,
        "status_counts": dict(status_counts),
        "identities_available": len(names),
    }
    np.savez_compressed(
        cache,
        embeddings=dataset["embeddings"],
        labels=dataset["labels"],
        paths=np.array(paths),
        bboxes=dataset["bboxes"],
        identity_names=np.array(kept_names),
        status_counts=json.dumps(dataset["status_counts"]),
        identities_available=len(names),
        settings_key=settings_key,
    )
    print(f"{label}: done — {dict(status_counts)}")
    return dataset


datasets = {
    "LFW": build_dataset("LFW", lfw_identities, LFW_DIR),
    "Bollywood": build_dataset("Bollywood", bollywood_identities, BOLLYWOOD_DIR),
}

## 7. Genuine and impostor scores

Every pair of usable images is compared with cosine similarity (dot product of unit vectors, as in
the backend). Same person → genuine pair; different people → impostor pair.

In [ ]:
def pair_scores(dataset: dict) -> dict:
    E, y = dataset["embeddings"], dataset["labels"]
    i, j = np.triu_indices(len(E), k=1)
    scores = np.einsum("ij,ij->i", E[i], E[j])
    same = y[i] == y[j]
    return {"scores": scores, "i": i, "j": j, "same": same,
            "genuine": scores[same], "impostor": scores[~same]}


pairs = {name: pair_scores(ds) for name, ds in datasets.items()}
for name, p in pairs.items():
    np.savez_compressed(OUT_DIR / f"{name.lower()}_scores.npz",
                        genuine=p["genuine"], impostor=p["impostor"])
    print(f"{name}: {len(p['genuine']):,} genuine pairs, {len(p['impostor']):,} impostor pairs")

## 8. Metrics

* **FAR** — fraction of impostor pairs scoring ≥ threshold (wrong person accepted).
* **FRR** — fraction of genuine pairs scoring < threshold (right person rejected).
* **EER** — the threshold where FAR ≈ FRR, found on a 0.001 grid exactly like the backend's
  `find_equal_error_rate`.
* **TAR@FAR** — share of genuine pairs accepted at the strictest threshold whose FAR ≤ target.
* **95% CI** — identity-level bootstrap (people resampled with replacement, 1,000 times).
  When zero false accepts are observed, the "rule of three" upper bound (3 / n) is also given.

In [ ]:
GRID = np.round(np.arange(0.0, 1.0 + THRESHOLD_STEP / 2, THRESHOLD_STEP), 3)
TAU_INDEX = int(round(MATCH_THRESHOLD / THRESHOLD_STEP))


def rates_on_grid(genuine: np.ndarray, impostor: np.ndarray):
    frr = np.searchsorted(np.sort(genuine), GRID, side="left") / len(genuine)
    far = 1.0 - np.searchsorted(np.sort(impostor), GRID, side="left") / len(impostor)
    return far, frr


def summarise(far: np.ndarray, frr: np.ndarray) -> dict:
    eer_index = int(np.argmin(np.abs(far - frr)))  # first minimum, like the backend's min()
    out = {
        "far_at_threshold": float(far[TAU_INDEX]),
        "frr_at_threshold": float(frr[TAU_INDEX]),
        "eer": float((far[eer_index] + frr[eer_index]) / 2),
        "eer_threshold": float(GRID[eer_index]),
    }
    for target in FAR_TARGETS:
        index = int(np.argmax(far <= target))  # far falls as the threshold rises
        out[f"tar_at_far_{target:g}"] = float(1.0 - frr[index])
        out[f"threshold_at_far_{target:g}"] = float(GRID[index])
    return out


N_BINS = int(round(2.0 / THRESHOLD_STEP))           # score histogram over [-1, 1)
GRID_BIN = np.round((GRID + 1.0) / THRESHOLD_STEP).astype(int)


def identity_bootstrap(dataset: dict, p: dict) -> dict:
    """Resamples people (not pairs). Per-person genuine histograms and per-person-pair impostor
    histograms make each replicate a weighted sum, so 1,000 replicates take seconds."""
    K = len(dataset["identity_names"])
    y = dataset["labels"]
    bins = np.clip(np.floor((p["scores"] + 1.0) / THRESHOLD_STEP + 1e-6).astype(np.int64),
                   0, N_BINS - 1)
    a, b, same = y[p["i"]], y[p["j"]], p["same"]

    G = np.bincount(a[same] * N_BINS + bins[same], minlength=K * N_BINS)
    G = G.reshape(K, N_BINS).astype(np.float32)
    lo, hi = np.minimum(a[~same], b[~same]), np.maximum(a[~same], b[~same])
    pair_index = lo * (2 * K - lo - 1) // 2 + (hi - lo - 1)   # position in np.triu_indices(K, 1)
    P = K * (K - 1) // 2
    H = np.bincount(pair_index * N_BINS + bins[~same], minlength=P * N_BINS)
    H = H.reshape(P, N_BINS).astype(np.float32)
    ia, ja = np.triu_indices(K, k=1)

    def rates(weights):
        g_cum = np.concatenate([[0.0], np.cumsum(weights @ G, dtype=np.float64)])
        i_cum = np.concatenate([[0.0], np.cumsum((weights[ia] * weights[ja]) @ H,
                                                 dtype=np.float64)])
        return (i_cum[-1] - i_cum[GRID_BIN]) / i_cum[-1], g_cum[GRID_BIN] / g_cum[-1]

    # Sanity check: with every person weighted once, the histograms must reproduce the exact result.
    check = summarise(*rates(np.ones(K, dtype=np.float32)))
    exact = summarise(*rates_on_grid(p["genuine"], p["impostor"]))
    assert all(abs(check[k] - exact[k]) < 1e-4 for k in exact), (check, exact)

    rng = np.random.default_rng(SEED)
    replicates = []
    for _ in range(N_BOOTSTRAP):
        weights = np.bincount(rng.integers(0, K, K), minlength=K).astype(np.float32)
        replicates.append(summarise(*rates(weights)))
    return {k: [float(np.percentile([r[k] for r in replicates], 2.5)),
                float(np.percentile([r[k] for r in replicates], 97.5))] for k in exact}


results = {}
for name, p in pairs.items():
    ds = datasets[name]
    far, frr = rates_on_grid(p["genuine"], p["impostor"])
    false_rejects = int((p["genuine"] < MATCH_THRESHOLD).sum())
    false_accepts = int((p["impostor"] >= MATCH_THRESHOLD).sum())
    results[name] = {
        "people": len(ds["identity_names"]),
        "people_available": ds["identities_available"],
        "images": len(ds["labels"]),
        "image_status_counts": ds["status_counts"],
        "genuine_pairs": len(p["genuine"]),
        "impostor_pairs": len(p["impostor"]),
        "confusion_at_threshold": {
            "genuine_accepted": len(p["genuine"]) - false_rejects,
            "genuine_rejected": false_rejects,
            "impostor_accepted": false_accepts,
            "impostor_rejected": len(p["impostor"]) - false_accepts,
        },
        "far_rule_of_three_upper": (3 / len(p["impostor"])) if false_accepts == 0 else None,
        "metrics": summarise(far, frr),
        "ci95": identity_bootstrap(ds, p),
        "genuine_score_mean": float(p["genuine"].mean()),
        "impostor_score_mean": float(p["impostor"].mean()),
        "_far": far,
        "_frr": frr,
    }
    m, ci = results[name]["metrics"], results[name]["ci95"]
    print(f"\n{name}  ({results[name]['people']} people, {results[name]['images']} images)")
    print(f"  at τ={MATCH_THRESHOLD}: FAR {m['far_at_threshold']:.4%} "
          f"[{ci['far_at_threshold'][0]:.4%}, {ci['far_at_threshold'][1]:.4%}]  "
          f"FRR {m['frr_at_threshold']:.4%} "
          f"[{ci['frr_at_threshold'][0]:.4%}, {ci['frr_at_threshold'][1]:.4%}]")
    print(f"  EER {m['eer']:.4%} [{ci['eer'][0]:.4%}, {ci['eer'][1]:.4%}] "
          f"at threshold {m['eer_threshold']:.3f}")
    for target in FAR_TARGETS:
        print(f"  TAR@FAR={target:g}: {m[f'tar_at_far_{target:g}']:.4%} "
              f"(threshold {m[f'threshold_at_far_{target:g}']:.3f})")

## 9. Figures (300 dpi, ready for the paper)

In [ ]:
import matplotlib

if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

COLORS = {"LFW": "#1f77b4", "Bollywood": "#d62728"}


def save(fig, filename):
    fig.savefig(OUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# 9a. DET curve — the main verification figure (ISO/IEC 19795-1): both error rates on log axes.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, r in results.items():
    far, frr = r["_far"], r["_frr"]
    visible = (far > 0) & (frr > 0)
    ax.plot(far[visible], frr[visible], color=COLORS[name], linewidth=2,
            label=f"{name} (EER {r['metrics']['eer']:.2%})")
    floor = 1 / r["impostor_pairs"]
    op_far = max(r["metrics"]["far_at_threshold"], floor)
    op_frr = max(r["metrics"]["frr_at_threshold"], 1 / r["genuine_pairs"])
    zero_note = " — 0 false accepts" if r["metrics"]["far_at_threshold"] == 0 else ""
    ax.scatter(op_far, op_frr, color=COLORS[name], marker="o", s=60, zorder=5,
               edgecolor="black", label=f"{name} at τ = {MATCH_THRESHOLD}{zero_note}")
eer_line = np.logspace(-7, 0, 50)
ax.plot(eer_line, eer_line, color="gray", linestyle=":", linewidth=1, label="FAR = FRR")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e-6, 1)
ax.set_ylim(1e-4, 1)
ax.set_xlabel("False Accept Rate (FAR)")
ax.set_ylabel("False Reject Rate (FRR)")
ax.set_title("DET curve — iPermit face verification")
ax.grid(which="both", alpha=0.3)
ax.legend(fontsize=8, loc="upper right")
save(fig, "fig_det_curve.png")

# 9b. Genuine vs impostor score distributions.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
bins = np.linspace(-0.3, 1.0, 131)
for ax, (name, p) in zip(axes, pairs.items()):
    r = results[name]
    ax.hist(p["impostor"], bins=bins, density=True, alpha=0.55, color="gray",
            label=f"Impostor pairs (n = {len(p['impostor']):,})")
    ax.hist(p["genuine"], bins=bins, density=True, alpha=0.65, color=COLORS[name],
            label=f"Genuine pairs (n = {len(p['genuine']):,})")
    ax.axvline(MATCH_THRESHOLD, color="black", linestyle="--", linewidth=1,
               label=f"Current threshold ({MATCH_THRESHOLD})")
    ax.axvline(r["metrics"]["eer_threshold"], color="green", linestyle=":", linewidth=1.5,
               label=f"EER threshold ({r['metrics']['eer_threshold']:.3f})")
    ax.set_title(f"{name} ({r['people']} people, {r['images']} images)")
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Genuine vs impostor similarity scores", fontweight="bold")
fig.tight_layout()
save(fig, "fig_score_distributions.png")

# 9c. FAR/FRR vs threshold (same style as the first run's figure, 0.001 resolution).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, (name, r) in zip(axes, results.items()):
    ax.plot(GRID, r["_far"], color="crimson", label="FAR")
    ax.plot(GRID, r["_frr"], color="steelblue", label="FRR")
    ax.axvline(MATCH_THRESHOLD, color="gray", linestyle="--",
               label=f"Current threshold ({MATCH_THRESHOLD})")
    eer_t = r["metrics"]["eer_threshold"]
    ax.axvline(eer_t, color="green", linestyle=":", label=f"EER threshold ({eer_t:.3f})")
    ax.plot(eer_t, r["metrics"]["eer"], "go", markersize=7)
    ax.set_title(f"{name}\n({r['genuine_pairs']:,} genuine, {r['impostor_pairs']:,} impostor pairs)")
    ax.set_xlabel("Cosine similarity threshold")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Error rate")
fig.suptitle("FAR / FRR vs threshold", fontweight="bold")
fig.tight_layout()
save(fig, "fig_far_frr_threshold.png")

## 10. Label-noise check — the lowest-scoring genuine pairs

Web-scraped celebrity datasets often contain photos of the **wrong person** under a name. Each such
photo counts as a false rejection and pushes FRR up. Look at these grids: if some pairs are clearly
two different people, part of the FRR is label noise, not a recognition error.

In [ ]:
def face_thumbnail(dataset: dict, index: int, size: int = 112):
    image = decode_image(dataset["root"] / dataset["paths"][index])  # original photo, no CLAHE
    x1, y1, x2, y2 = dataset["bboxes"][index]
    mw, mh = 0.25 * (x2 - x1), 0.25 * (y2 - y1)
    crop = image[max(0, int(y1 - mh)):int(y2 + mh), max(0, int(x1 - mw)):int(x2 + mw)]
    return cv2.cvtColor(cv2.resize(crop, (size, size)), cv2.COLOR_BGR2RGB)


def lowest_genuine_grid(name: str, count: int = 20, per_row: int = 4):
    ds, p = datasets[name], pairs[name]
    genuine_i, genuine_j = p["i"][p["same"]], p["j"][p["same"]]
    order = np.argsort(p["genuine"])[:count]
    rows = int(np.ceil(len(order) / per_row))
    fig, axes = plt.subplots(rows, per_row * 2, figsize=(per_row * 4, rows * 2.4))
    axes = np.atleast_2d(axes)
    for ax in axes.flat:
        ax.axis("off")
    for k, pair in enumerate(order):
        row, col = divmod(k, per_row)
        a, b = genuine_i[pair], genuine_j[pair]
        axes[row, col * 2].imshow(face_thumbnail(ds, a))
        axes[row, col * 2 + 1].imshow(face_thumbnail(ds, b))
        person = ds["identity_names"][ds["labels"][a]]
        axes[row, col * 2].set_title(f"{person[:18]}\nscore {p['genuine'][pair]:.3f}",
                                     fontsize=8, loc="left")
    fig.suptitle(f"{name}: {len(order)} lowest-scoring genuine pairs "
                 f"(threshold {MATCH_THRESHOLD})", fontweight="bold")
    fig.tight_layout()
    save(fig, f"fig_lowest_genuine_pairs_{name.lower()}.png")


for name in datasets:
    lowest_genuine_grid(name)

## 11. Per-person error table

Shows whether false rejections are spread across many people or concentrated in a few
(a few people with very high FRR usually points to mislabelled photos).

In [ ]:
import csv

for name, ds in datasets.items():
    p = pairs[name]
    person = ds["labels"][p["i"][p["same"]]]
    rows = []
    for label, identity in enumerate(ds["identity_names"]):
        scores = p["genuine"][person == label]
        rejected = int((scores < MATCH_THRESHOLD).sum())
        rows.append({"person": identity, "images": int((ds["labels"] == label).sum()),
                     "genuine_pairs": len(scores), "false_rejects": rejected,
                     "frr": rejected / len(scores), "mean_genuine_score": float(scores.mean()),
                     "min_genuine_score": float(scores.min())})
    rows.sort(key=lambda r: r["frr"], reverse=True)
    with open(OUT_DIR / f"per_person_frr_{name.lower()}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    with_errors = sum(r["false_rejects"] > 0 for r in rows)
    print(f"\n{name}: {with_errors} of {len(rows)} people have at least one false rejection")
    for r in rows[:10]:
        print(f"  {r['person'][:28]:28s} FRR {r['frr']:6.1%}  "
              f"({r['false_rejects']}/{r['genuine_pairs']})  min score {r['min_genuine_score']:.3f}")

## 12. Save the results and download

In [ ]:
import datetime
import platform

import insightface


def pct(value):
    return f"{value:.3%}"


def pct_ci(r, key):
    low, high = r["ci95"][key]
    return f"{pct(r['metrics'][key])} [{pct(low)}–{pct(high)}]"


lines = [
    "# iPermit face verification — results",
    "",
    f"Pipeline: CLAHE → insightface buffalo_l (RetinaFace + ArcFace w600k_r50), det_size 640×640. "
    f"Up to {MAX_IMAGES_PER_IDENTITY} usable images per person, seed {SEED}. "
    f"95% CIs: identity-level bootstrap, {N_BOOTSTRAP} replicates.",
    "",
    "## Table 1 — Datasets",
    "",
    "| Dataset | People | Usable images | Skipped (0 faces / 2+ faces / unreadable / duplicate) "
    "| Genuine pairs | Impostor pairs |",
    "|---|---|---|---|---|---|",
]
for name, r in results.items():
    s = r["image_status_counts"]
    lines.append(
        f"| {name} | {r['people']} | {r['images']:,} | {s.get('no_face', 0)} / "
        f"{s.get('multiple_faces', 0)} / {s.get('unreadable', 0)} / {s.get('duplicate_file', 0)} "
        f"| {r['genuine_pairs']:,} | {r['impostor_pairs']:,} |")
lines += [
    "",
    f"## Table 2 — Verification performance (95% CI)",
    "",
    f"| Dataset | FAR @ τ={MATCH_THRESHOLD} | FRR @ τ={MATCH_THRESHOLD} | EER | EER threshold | "
    + " | ".join(f"TAR @ FAR={t:g}" for t in FAR_TARGETS) + " |",
    "|---|---|---|---|---|" + "---|" * len(FAR_TARGETS),
]
for name, r in results.items():
    m = r["metrics"]
    far_cell = pct_ci(r, "far_at_threshold")
    if r["far_rule_of_three_upper"] is not None:
        far_cell = f"0 (≤ {pct(r['far_rule_of_three_upper'])}, rule of three)"
    lines.append(
        f"| {name} | {far_cell} | {pct_ci(r, 'frr_at_threshold')} | {pct_ci(r, 'eer')} | "
        f"{m['eer_threshold']:.3f} | "
        + " | ".join(pct_ci(r, f"tar_at_far_{t:g}") for t in FAR_TARGETS) + " |")
lines += [
    "",
    f"## Table 3 — Confusion counts at τ = {MATCH_THRESHOLD}",
    "",
    "| Dataset | Genuine accepted (TP) | Genuine rejected (FN) | Impostor accepted (FP) "
    "| Impostor rejected (TN) |",
    "|---|---|---|---|---|",
]
for name, r in results.items():
    c = r["confusion_at_threshold"]
    lines.append(f"| {name} | {c['genuine_accepted']:,} | {c['genuine_rejected']:,} | "
                 f"{c['impostor_accepted']:,} | {c['impostor_rejected']:,} |")
(OUT_DIR / "results_tables.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or None
except FileNotFoundError:
    gpu = None
metrics = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "config": {
        "seed": SEED, "max_images_per_identity": MAX_IMAGES_PER_IDENTITY,
        "lfw_min_images_per_identity": LFW_MIN_IMAGES_PER_IDENTITY,
        "match_threshold": MATCH_THRESHOLD, "threshold_step": THRESHOLD_STEP,
        "n_bootstrap": N_BOOTSTRAP, "clahe": True, "det_size": [640, 640],
        "max_identities_override": MAX_IDENTITIES,
    },
    "environment": {
        "python": platform.python_version(), "insightface": insightface.__version__,
        "onnxruntime": ort.__version__, "opencv": cv2.__version__, "numpy": np.__version__,
        "execution_provider": ACTIVE_PROVIDER, "gpu": gpu,
    },
    "results": {name: {k: v for k, v in r.items() if not k.startswith("_")}
                for name, r in results.items()},
}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

archive_path = WORK_DIR / "ipermit_face_eval_results.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for f in sorted(OUT_DIR.iterdir()):
        archive.write(f, f.name)
print(f"\nSaved {archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files

    files.download(str(archive_path))